## Datasets Used

### 1. MERIT-Basins

**Description**  
A global reconstruction of naturalized river flows for approximately **2.94 million river reaches**, derived from the MERIT hydrography framework. The dataset provides a consistent global representation of river connectivity and flow characteristics suitable for large-scale hydrological modeling.

> **Note:** A bug-fixed version of this dataset was used in the present work.

**Citation**  
Lin, P., Pan, M., Beck, H. E., Yang, Y., Yamazaki, D., Frasson, R., *et al.* (2019).  
*Global reconstruction of naturalized river flows at 2.94 million reaches.*  
**Water Resources Research**, 55(8), 6499–6516.  
https://doi.org/10.1029/2019WR025287

**Dataset Access**  
- MERIT-Basins Hydrography (based on MERIT-Hydro v0.7 / v1.0_bugfix1):  
  https://www.reachhydro.org/home/params/merit-basins  
  *(Includes minor bug fixes for coastline pixels)*

---

### 2. HydroLAKES (Version 1)

**Description**  
A global vector database of lakes and reservoirs, providing detailed information on lake shorelines, surface area, volume, depth estimates, and hydrological connectivity. HydroLAKES is widely used in global hydrology and water resources studies.

**Citation**  
Messager, M. L., Lehner, B., Grill, G., Nedeva, I., & Schmitt, O. (2016).  
*Estimating the volume and age of water stored in global lakes using a geostatistical approach.*  
**Nature Communications**, 7, 13603.  
https://doi.org/10.1038/ncomms13603

**Dataset Access**  
- HydroLAKES product page:  
  https://www.hydrosheds.org/products/hydrolakes

### Assigning parameters and folders

In [1]:
# outputfolder for where the files will be sitting
OutFolder = '/Users/shg096/Desktop/LakeRiverOut/MERITBasins/'

regions = {
    "71": {
        "files": {
            "riv": "/Users/shg096/Desktop/MERIT_Hydro/riv/riv_pfaf_71_MERIT_Hydro_v07_Basins_v01_bugfix1.shp",
            "cat": "/Users/shg096/Desktop/MERIT_Hydro/cat/cat_pfaf_71_MERIT_Hydro_v07_Basins_v01_bugfix1.shp",
            "cst": "/Users/shg096/Desktop/MERIT_Hydro/hill/hillslope_71_clean.shp",
        }
    }
}

# combinations of flags
force_options = [True, False]
single_options = [False, True]
suffixes = ["a", "b", "c", "d"]

# location of HydroLAKES
lake_file = '/Volumes/F:/hydrography/hydrolakes/HydroLAKES_polys_v10_shp/HydroLAKES_polys_v10_shp/HydroLAKES_polys_v10.shp'

In [5]:
# load the needed packages
import os
import shutil
import geopandas as gpd
from   itertools import product
from   riverlakenetwork import Utility, BurnLakes
import warnings; warnings.filterwarnings("ignore")

In [3]:
#load hydrolakeDataset
lake = gpd.read_file(lake_file) # read the hydrolake dataset
# merge lake Michigan and Huron as they are hydraulically connected
lake = Utility.FixHydroLAKESv1(lake, merge_lakes={"Michigan+Huron": [6, 8]})
lake = lake.set_crs("EPSG:4326") # make sure the lake has projection

In [6]:
# ----------------------------------
# Loop
# ----------------------------------
for pfaf, region_data in regions.items():

    files = region_data["files"]

    for (force_flag, single_flag), suffix in zip(product(force_options, single_options), suffixes):

        pfaf_name = f"{pfaf}{suffix}"
        print(f"Running {pfaf_name} | force={force_flag} | single={single_flag}")

        # ----------------------------------
        # Read data
        # ----------------------------------
        riv, cat = Utility.merit_read_file(
            riv_file=files["riv"],
            cat_file=files["cat"],
            cst_file=files["cst"]
        )

        # ----------------------------------
        # Output folders
        # ----------------------------------
        def reset_folder(path):
            if os.path.isdir(path):
                shutil.rmtree(path)
            os.makedirs(path, exist_ok=True)

        org_folder = os.path.join(OutFolder, f"pfaf{pfaf_name}_org")
        corrected_folder = os.path.join(OutFolder, f"pfaf{pfaf_name}_corrected")

        reset_folder(org_folder)
        reset_folder(corrected_folder)

        # ----------------------------------
        # Manual fix (only if needed)
        # ----------------------------------
        riv.loc[riv["COMID"] == 74030207,
                ["NextDownID", "geometry", "maxup", "up1", "lengthkm"]] = [-9999, None, 0, 0, 0]

        # ----------------------------------
        # CRS
        # ----------------------------------
        riv = riv.set_crs("EPSG:4326")
        cat = cat.set_crs("EPSG:4326")

        # ----------------------------------
        # Save original
        # ----------------------------------
        riv.to_file(os.path.join(org_folder, "riv.gpkg"))
        cat.to_file(os.path.join(org_folder, "cat.gpkg"))

        # ----------------------------------
        # Config
        # ----------------------------------
        config = {
            "riv": riv,
            "riv_dict": {
                "COMID": {"col": "COMID"},
                "NextDownCOMID": {"col": "NextDownID"},
                "length": {"col": "lengthkm"},
                "uparea": {"col": "uparea", "unit": "km2"},
            },
            "cat": cat,
            "cat_dict": {
                "COMID": {"col": "COMID"},
                "unitarea": {"col": "unitarea", "unit": "km2"},
            },
            "lake": lake,
            "lake_dict": {
                "LakeCOMID": {"col": "Hylak_id"},
                "unitarea": {"col": "Lake_area", "unit": "km2"},
            },
        }

        # ----------------------------------
        # Run model
        # ----------------------------------
        bl = BurnLakes(
            config=config,
            force_one_lake_per_riv_seg_flag=force_flag,
            single_segment_lakes_activate_flag=single_flag
        )

        # ----------------------------------
        # Save outputs
        # ----------------------------------
        bl.riv.to_file(os.path.join(corrected_folder, "riv.gpkg"))
        bl.cat.to_file(os.path.join(corrected_folder, "cat.gpkg"))
        bl.lake.to_file(os.path.join(corrected_folder, "lake.gpkg"))

Running 71a | force=True | single=False
=== Input loader started at : 2026-06-23 18:59:37  ===
riv: Loaded
riv_dict: {'COMID': {'col': 'COMID'}, 'NextDownCOMID': {'col': 'NextDownID'}, 'length': {'col': 'lengthkm'}, 'uparea': {'col': 'uparea', 'unit': 'km2'}}
cat: Loaded
cat_dict: {'COMID': {'col': 'COMID'}, 'unitarea': {'col': 'unitarea', 'unit': 'km2'}}
lake: Loaded
lake_dict: {'LakeCOMID': {'col': 'Hylak_id'}, 'unitarea': {'col': 'Lake_area', 'unit': 'km2'}}
=== Input loader finished at: 2026-06-23 18:59:38  ===
=== Input loader took      : 0:00:00.859701  ===========================
=== Input checker started at : 2026-06-23 18:59:38  ===
Subbasin and lake area units are consistent: km2
riv CRS: EPSG:4326
cat CRS: EPSG:4326
lake CRS: EPSG:4326
✅ No loop detected in network topology
=== Input checker finished at: 2026-06-23 18:59:49  ===
=== Input checker took      : 0:00:10.764584  ===========================
=== Resolving lakes started at : 2026-06-23 18:59:49  ===
==== Number of l